<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/MLTestSuccess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb


drive.mount('/content/drive')
path = "/content/drive/MyDrive/sparcs_model_v1.feather"
data = pd.read_feather(path)



#Categorical columns
categorical_cols = ['zip_code', 'health_service_area', 'facility_id', 'hospital_county', 'age_group',
                    'gender', 'race', 'ethnicity', 'admission_type', 'apr_mortality_risk',
                    'apr_severity_code', 'apr_drg_code', 'apr_mdc_code', 'ccsr_dx_code']
#Numerical columns
numeric_cols = ['payer_medicaid', 'payer_medicare','payer_private_insurance','payer_self_pay',
                'payer_blue_cross','payer_other', 'payer_gov_va',
                'payer_corrections','payer_managed_care','num_payment_types']



#Convert categorical columns to category dtype
for col in categorical_cols:
    data[col] = data[col].astype('category')

# Convert numerical and drop missing data/zip codes
data['length_of_stay'] = pd.to_numeric(data['length_of_stay'], errors='coerce')
data = data.dropna(subset=['length_of_stay'])

if 'zip_code' in data.columns:
    data = data[data['zip_code'] != 'OOS']



# Define features and target
X = data[categorical_cols + numeric_cols]
y = data['los_log']

# LightGBM Regressor (memory-efficient and fast)
model = lgb.LGBMRegressor(n_estimators = 1200, learning_rate = 0.03, num_leaves = 63,
                          max_depth = -1, subsample = 0.8, colsample_bytree = 0.8,
                          n_jobs = -1, random_state = 42)
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)



# Evaluate
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")

#First Run Data Results:
#MAE: 3.91
#RMSE: 7.97
#R2: 0.172

#metrics indicate the model is struggling, typical for LOS (length of stay) prediction,
#but the R2 = 0.17 means the model only explains 17% of variance
#model runs, just needed to add other affecting variables to increase the models functioning on determining LOS :)



#RE-RUN with improved code:
#MAE: 0.37
#RMSE: 0.50
#R2: 0.527

#model is capturing far more variance R2 = 52.7%


Mounted at /content/drive
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.466120 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 969
[LightGBM] [Info] Number of data points in the train set: 3288528, number of used features: 24
[LightGBM] [Info] Start training from score 1.576022
MAE: 0.37
RMSE: 0.50
R2: 0.527
